
# Prediction and Applicability Domain

This notebook evaluates the applicability domain (AD) of a QSAR classification model developed to predict earthworm acute toxicity.

In [ ]:
import pandas as pd
from sklearn.metrics import recall_score, accuracy_score, precision_score, confusion_matrix
from sklearn.model_selection import cross_validate


results={}
def check_classification_model(model,train_file,test_file,name,save=True):

    #First, open the training and test files, and save the dataframes containing the descriptors and endpoint values.
    train_dataset = pd.read_csv(train_file,sep=';')
    X_train = train_dataset.iloc[:, 2:] # Assuming columns from index 2 onwards are features
    Y_train = train_dataset.iloc[:, 1] # Select the second column (index 1) as the target variable ('y')

    # Check if X_train is empty
    if X_train.empty:
        raise ValueError(f"Training feature matrix (X_train) is empty. Please check the content and column structure of '{train_file}'. It might contain only one column (the target) or no data columns.")

    test_dataset = pd.read_csv(test_file,sep=';')
    X_test = test_dataset.iloc[:, 2:] # Assuming columns from index 2 onwards are features
    Y_test = test_dataset.iloc[:, 1] # Select the second column (index 1) as the target variable ('y')

    # Check if X_test is empty
    if X_test.empty:
        raise ValueError(f"Test feature matrix (X_test) is empty. Please check the content and column structure of '{test_file}'. It might contain only one column (the target) or no data columns.")

    #Fit the model using the training set only
    model.fit(X_train,Y_train)

    #Make predictions for the training set and calculate the corresponding metrics
    Y_train_pred = model.predict(X_train)
    tr_Accuracy=round(accuracy_score(Y_train,Y_train_pred),2)
    tr_Sensitivity=round(recall_score(Y_train,Y_train_pred),2)
    tr_Specificity=round(recall_score(Y_train,Y_train_pred,pos_label=0),2)
    tr_Precision=round(precision_score(Y_train,Y_train_pred),2)
    tr_cm=confusion_matrix(Y_train,Y_train_pred)

    #Make predictions for the test set and calculate the corresponding metrics
    Y_test_pred = model.predict(X_test)
    ts_Accuracy=round(accuracy_score(Y_test,Y_test_pred),2)
    ts_Sensitivity=round(recall_score(Y_test,Y_test_pred),2)
    ts_Specificity=round(recall_score(Y_test,Y_test_pred,pos_label=0),2)
    ts_Precision=round(precision_score(Y_test,Y_test_pred),2)
    ts_cm=confusion_matrix(Y_test,Y_test_pred)

    #Perform cross-validation and save only the test accuracy
    cv_results = cross_validate(model, X_train, Y_train,scoring='accuracy',return_train_score=True)
    cv_Accuracy_mean=round(cv_results['test_score'].mean(),3)
    cv_Accuracy_dev=round(cv_results['test_score'].std(),3)

    if save:
      results.update({name:[tr_Accuracy,tr_Sensitivity,tr_Specificity,tr_Precision,
            ts_Accuracy,ts_Sensitivity,ts_Specificity,ts_Precision,
            cv_Accuracy_mean,cv_Accuracy_dev]})

    #Print the results
    print('        TRAIN\t\t\t\tTEST')
    print('\t\tNeg.\tPos.\t\t\tNeg.\tPos.')
    print('\tNegative\t'+str(tr_cm[0][0])+'\t'+str(tr_cm[0][1])+'\t\tNegative\t'+str(ts_cm[0][0])+'\t'+str(ts_cm[0][1]))
    print('\tPositive\t'+str(tr_cm[1][0])+'\t'+str(tr_cm[1][1])+'\t\tPositive\t'+str(ts_cm[1][0])+'\t'+str(ts_cm[1][1]))
    print('Accuracy:\t',tr_Accuracy,'\t\t\t\t',ts_Accuracy)
    print('Sensitivity:\t',tr_Sensitivity,'\t\t\t\t',ts_Sensitivity)
    print('Specificity:\t',tr_Specificity,'\t\t\t\t',ts_Specificity)
    print('Precision:\t',tr_Precision,'\t\t\t\t',ts_Precision)
    print('Cross validation test accuracy:',cv_Accuracy_mean,'+/-',cv_Accuracy_dev)

    return [tr_Accuracy,tr_Sensitivity,tr_Specificity,tr_Precision,
            ts_Accuracy,ts_Sensitivity,ts_Specificity,ts_Precision,
            cv_Accuracy_mean,cv_Accuracy_dev]

Apply the function to the model you selected to perform the predictions.

In [ ]:
#@title Train the model
# WRITE YOUR CODE HERE

import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression  #Import the selected algorithm

# Instantiate LogisticRegression with default parameters
estimator = LogisticRegression(random_state=42, max_iter=1000)

# Helper to map text labels and align common features between training and test sets
def preprocess_and_align_datasets(train_path, test_path, out_train_path, out_test_path):
    df_train = pd.read_csv(train_path, sep=';')
    df_test = pd.read_csv(test_path, sep=';')

    # Clean and map training target labels
    df_train.iloc[:, 1] = df_train.iloc[:, 1].astype(str).str.strip().str.lower()
    df_train.iloc[:, 1] = df_train.iloc[:, 1].map({'toxic': 1, 'non-toxic': 0, '1': 1, '0': 0})

    # Clean and map test target labels
    df_test.iloc[:, 1] = df_test.iloc[:, 1].astype(str).str.strip().str.lower()
    df_test.iloc[:, 1] = df_test.iloc[:, 1].map({'toxic': 1, 'non-toxic': 0, '1': 1, '0': 0})

    # Drop rows where target variable is NaN
    df_train = df_train.dropna(subset=[df_train.columns[1]])
    df_test = df_test.dropna(subset=[df_test.columns[1]])

    # Find common features (columns starting from index 2)
    features_train = set(df_train.columns[2:])
    features_test = set(df_test.columns[2:])
    common_features = sorted(list(features_train.intersection(features_test)))

    if not common_features:
        # If no overlap, we default to the training set's feature structure and force-align the test set
        common_features = list(df_train.columns[2:])
        for col in common_features:
            if col not in df_test.columns:
                df_test[col] = 0.0

    # Filter to only keep the ID, label, and matched features
    keep_cols = list(df_train.columns[:2]) + common_features

    df_train_aligned = df_train[keep_cols]
    df_test_aligned = df_test[keep_cols]

    df_train_aligned.to_csv(out_train_path, sep=';', index=False)
    df_test_aligned.to_csv(out_test_path, sep=';', index=False)

temp_train_path = '/content/sample_data/temp_train_encoded.csv'
temp_test_path = '/content/sample_data/temp_test_encoded.csv'

preprocess_and_align_datasets(
    '/content/sample_data/Earthworm_acute_toxicity_Train_RFE_logreg (1) (1).csv',
    '/content/sample_data/Earthworm_acute_toxicity_Test_RFE_logreg (1) (1).csv',
    temp_train_path,
    temp_test_path
)

# Use the estimator for check_classification_model with the aligned datasets
check_classification_model(estimator, # The estimator
                           temp_train_path, #The training file name
                           temp_test_path,  #The test file name
                           'Selected_model')  #A name for saving the results

In [ ]:
#@title Predict the toxicity of caffeine towards earthworms
#Write your code here

# Install required chemical calculation packages
!pip install rdkit mordred -q

import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors
from mordred import Calculator, descriptors
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler

# Define the missing imputer_scaler function
def imputer_scaler(df_target, df_reference):
    imputer = KNNImputer(missing_values=np.nan, n_neighbors=3, weights="uniform")
    scaler = StandardScaler()

    columns_descriptors = list(df_reference.columns)
    reference_descriptors_matrix = df_reference[columns_descriptors]

    imputer.fit(reference_descriptors_matrix)
    target_descriptors_matrix = df_target[columns_descriptors]

    imputed_ref_matrix = imputer.transform(reference_descriptors_matrix)
    imputed_matrix = imputer.transform(target_descriptors_matrix)

    scaler.fit(imputed_ref_matrix)
    imputed_scaled_matrix = scaler.transform(imputed_matrix)

    df_imputed_scaled = pd.DataFrame(imputed_scaled_matrix, columns=columns_descriptors)
    return df_imputed_scaled

# 1. Obtain the chemical structure of caffeine
smiles_caffeine = "CN1C=NC2=C1C(=O)N(C(=O)N2C)C"
mol_caffeine = Chem.MolFromSmiles(smiles_caffeine)

# 2. Calculate its RDKit and Mordred descriptors
rdkit_desc = Descriptors.CalcMolDescriptors(mol_caffeine)
df_rdkit = pd.DataFrame([rdkit_desc])

calc = Calculator(descriptors, ignore_3D=True)
mordred_df = calc.pandas([mol_caffeine])

df_all_desc = pd.concat([pd.DataFrame({'SMILES': [smiles_caffeine]}), df_rdkit, mordred_df], axis=1)

# 3. Align features with the training set columns
train_database = pd.read_csv('/content/sample_data/temp_train_encoded.csv', sep=';')
train_descs = train_database.columns[2:].tolist()

X_caf = pd.DataFrame(columns=train_descs, index=[0])
for col in train_descs:
    if col in df_all_desc.columns:
        X_caf[col] = df_all_desc[col].iloc[0]

# 4. Perform imputation and scaling (from training set reference)
train_database_reduced = train_database[train_descs]
X_caf_scaled = imputer_scaler(X_caf, train_database_reduced)

# 5. Predict the toxicity using the estimator
Y_caf_pred = estimator.predict(X_caf_scaled)

#Keep this to view the results
Y_caf_pred

  Preparing metadata (setup.py) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.8/31.8 MB 56.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 57.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
mapclassify 2.10.0 requires networkx>=3.2, but you have networkx 2.8.8 which is incompatible.
rasterio 1.5.2 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
jax 0.11.1 requires numpy>=2.1, but you have numpy 1.26.4 which is incompatible.
shap 0.52.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-python 5.0.0.93 requi

ImportError: cannot import name 'product' from 'numpy' (/usr/local/lib/python3.13/dist-packages/numpy/__init__.py)

###**Q1.** What is the predicted toxicity of caffeine?

How confident are we in this prediction? A first approach is to check the probability assigned by the machine learning algorithm itself. Use the predict_proba method to obtain the probabilities of being negative (first value) and positive (second value).



In [ ]:
#@title Prediction probability
#Write your code here
Y_caf_prob = estimator.predict_proba(X_caf_scaled)


#Keep this to view the results
Y_caf_prob

array([[0.0108661, 0.9891339]])

**Note:** If you selected a Support Vector Machine (SVM) as your algorithm, this calculation is not available by default. It is possible to enable probability estimation (by setting it during the fitting process), but this is not necessary here.

## Applicability Domain of caffeine in the earthworm model

A prediction has been made — but can we trust it? In addition to assessing the overall quality of the model, we need to verify whether the molecule in question lies within the chemical space covered by our model. To do this, we measure the applicability domain.

There are several methods to assess it, including Leverage, Euclidean Distance, Tanimoto Similarity, and Kernel Density Estimation.



## Section 3: Tanimoto index

The Tanimoto index is a measure of molecular similarity based on fingerprints. Therefore, the first thing we need is a fingerprint.

A fingerprint is an identifier represented by a binary string indicating the presence or absence of specific groups or features within a molecule. Since there is no single universal set of features, several types of fingerprints can be used. In this case, we will use the MACCS Keys fingerprint.

These can be calculated using the rdMolDescriptors module from RDKit (see the [documentation](https://www.rdkit.org/docs/source/rdkit.Chem.rdMolDescriptors.html)).

Obtain the MACCS Keys fingerprint for caffeine.


In [ ]:
#@title Obtain the fingerprint of caffeine
from rdkit.Chem import rdMolDescriptors

#Get the MACCS fingerprint using the mol object created earlier
fingerprint = rdMolDescriptors.GetMACCSKeysFingerprint(mol)


#Keep the print statements to check your work
print("Fingerprint should be an ExplicitBitVect object:",fingerprint)
print("This object can be expanded to a list of 0/1 values:",list(fingerprint))

Fingerprint should be an ExplicitBitVect object: <rdkit.DataStructs.cDataStructs.ExplicitBitVect object at 0x7fea963d4040>
This object can be expanded to a list of 0/1 values: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0]


As you can see, the fingerprint is stored as a binary object for efficiency, but it can be converted into a list to check the individual 0 and 1 values.

Next, we need to compare this fingerprint with those in our training dataset. We will use the Jaccard–Tanimoto coefficient for similarity, which is calculated as:

$$S_{J-T} = \frac{P_{both}}{P_{both}+P_{1}+P_{2}} $$
where P represents the number of features that are positive in both molecules, only in the first, and only in the second (${P_{both}}$, $P_{1}$, and $P_{2}$, respectively).

As a test, let’s calculate the similarity index of caffeine with butyric acid and nicotine using the formula above. Note that the easiest way to operate with the 0s and 1s of the index is to convert the fingerprint into a list beforehand.



In [ ]:
#@title Manual calculation of the Tanimoto-Jaccard Similariy index
#Complete the code
but_mol = Chem.MolFromSmiles("O=C(O)CCC")
butiric = rdMolDescriptors.GetMACCSKeysFingerprint(but_mol)
nico_mol = Chem.MolFromSmiles("CN1CCC(C[N+]1C)C1=CN=CC=C1")
nicotine = rdMolDescriptors.GetMACCSKeysFingerprint(nico_mol)
lbutiric = list(butiric)
lnicotine = list(nicotine)
lfingerprint = list(fingerprint)

p_both = 0
p_1 = 0
p_2 = 0
for i,j in enumerate(fingerprint):
    if  lfingerprint[i]== 1 and lnicotine[i]==1:
        p_both = p_both+1
    elif lfingerprint[i]== 1 and lnicotine[i]==0:
        p_1 = p_1+1
    elif lfingerprint[i]== 0 and lnicotine[i]==1:
        p_2 = p_2+1

sim_nico = p_both / (p_both+p_1+p_2)
p_both = 0
p_1 = 0
p_2 = 0
for i,j in enumerate(fingerprint):
    if  lfingerprint[i]== 1 and lbutiric[i]==1:
        p_both = p_both+1
    elif lfingerprint[i]== 1 and lbutiric[i]==0:
        p_1 = p_1+1
    elif lfingerprint[i]== 0 and lbutiric[i]==1:
        p_2 = p_2+1

sim_buti = p_both / (p_both+p_1+p_2)

#Keep the print statements to check your work
print("Fingerprint of caffeine:",list(fingerprint))
print("Fingerprint of nicotine:",list(nicotine))
print("Fingerprint of butiric acid:",list(butiric))
print("Similarity index for nicotine (should be 0.386):",round(sim_nico,3))
print("Similarity index for butiric acid (should be 0.070):",round(sim_buti,3))

Fingerprint of caffeine: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0]
Fingerprint of nicotine: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 1, 0, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 1, 

On the other hand, RDKit provides a function, `FingerprintSimilarity`, which calculates the index directly from the binary vectors. Use this function and verify that it produces the same values.

In [ ]:
from rdkit import DataStructs

sim_nico = DataStructs.FingerprintSimilarity(fingerprint,nicotine)
sim_buti = DataStructs.FingerprintSimilarity(fingerprint,butiric)

print("Similarity index for nicotine (should be 0.386):",round(sim_nico,3))
print("Similarity index for butiric acid (should be 0.070):",round(sim_buti,3))


Similarity index for nicotine (should be 0.386): 0.297
Similarity index for butiric acid (should be 0.070): 0.07


To check the chemical space of a database, we need to calculate the similarity index of our target molecule with each molecule in the training dataset. For each one, we will obtain the fingerprint and calculate the index.

To determine whether caffeine lies within our applicability domain, it must be sufficiently similar to at least one molecule in the training set. Therefore, we only need to focus on the most similar molecule — that is, the one with the highest similarity value.

As a result, we do not need to store all the individual values; we only need to keep track of the maximum similarity value and report it at the end.

In [ ]:
#@title Check the Tanimoto similarity for the prediction

#Import the necessary modules, if needed
from rdkit import Chem
from rdkit.Chem import rdMolDescriptors
from rdkit import DataStructs

#To avoid mistakes, it is best to include the caffeine SMILES again and recreate the Mol object
smiles = "CN1C=NC2=C1C(=O)N(C(=O)N2C)C" #Introduce the SMILES
mol = Chem.MolFromSmiles(smiles)
target_fingerprint = rdMolDescriptors.GetMACCSKeysFingerprint(mol)


#You can use the SMILES column from the train_database dataframe opened earlier
smi_train = train_database['SMILES']

#Define a variable to keep track of the maximum value and initialise it with a low value (e.g., 0.0)
max_sim = 0.00
#Loop through the SMILES in the training dataset
for i in smi_train:
    #Obtain the Mol object and fingerprint for each molecule in the training dataset
    train_mol = Chem.MolFromSmiles(i)
    if train_mol is not None: # Ensure mol object is created successfully
        train_fingerprint = rdMolDescriptors.GetMACCSKeysFingerprint(train_mol)

        #Calculate the similarity score using the FingerprintSimilarity function
        sim_score = DataStructs.FingerprintSimilarity(target_fingerprint, train_fingerprint)
        #Update the variable if the new value is greater than the current maximum
        if sim_score > max_sim:
            max_sim = sim_score

print('Maximum similarity index:',round(max_sim,3))

Maximum similarity index: 0.561


The similarity value that determines inclusion within the applicability domain depends on the fingerprint used. For example, for MACCS fingerprints, an acceptable threshold is 0.573, which corresponds to a similarity of more than 95%.

For more information, see: http://rdkit.blogspot.com/2013/10/fingerprint-thresholds.html

###**Q2.** What is the value of the maximum similarity Tanimoto-Jaccard index for the caffeine in your dataset?

## Section 4: Leverage

In statistics, leverage is a measure of how different each compound is from the rest of the group, based on the values of its descriptors. Below, you will find a function that calculates this measure. The mathematical details are somewhat complex, so we will not discuss them here.

In [ ]:
#@title Leverage function
import numpy as np

def leverage(X_train,X_target):
        #Ensure that the training descriptors matrix is of float type
        X_train = X_train.astype(float)

        # The h value for each descriptor row in the dataset is defined as: hii = xi T(XTX)–1 xi
        # Calculate (XTX)–1
        tranpX = np.transpose(X_train)
        xtx = np.dot(tranpX, X_train)
        invxtx = np.linalg.inv(xtx)


        leverages_target = []
        leverages_in_out = []

        for idx, row in X_target.iterrows():
            transposed_row = np.transpose(row)
            leverage = np.dot(np.dot(row, invxtx), transposed_row)
            leverages_target.append(leverage.round(4))

        return leverages_target



In addition to structural differences, leverage values strongly depend on both the number of molecules included in the training set and the number of descriptors. Therefore, the threshold used to determine whether a molecule is similar enough to be included in the applicability domain depends on these two factors and is calculated as:

$$
3 \times \left( \frac{\text{number of descriptors}}{\text{number of molecules}} \right)
$$

In [ ]:
#@title Calculate the threshold and determine whether the molecules are inside or outside the applicability domain

X_train_scaled = imputer_scaler(database_reduced, database_reduced) #Create a matrix containing only the scaled descriptors

#Calculate the leverage threshold
p = X_train_scaled.shape[1]  #p is the number of descriptors
n = X_train_scaled.shape[0]  #n is the number of molecules in the training set
warning_leverage = 3*(p/n)

leverages = leverage(X_train_scaled,X_caf_scaled)

leverages_in_out = []
for value in leverages:
    if value < warning_leverage:
            leverages_in_out.append('IN')
    else:
            leverages_in_out.append('OUT')

## Section 5: Euclidean distance

Another measure of molecular similarity that can be used to assess the applicability domain is the Euclidean distance. The Euclidean distance is the standard distance measure used in geometry, but in this case it is extended to n dimensions, where each dimension corresponds to one of the descriptors used in the model.

The formula is:

$d(p,q) = \sqrt{(p_1+q_1)^2+(p_2+q_2)^2+...+(p_n+q_n)^2} $, for n dimensions, where $p_i$ and $q_i$ are the standardised values of descriptor i for each molecule.

Because the magnitude of the Euclidean distances depends on the descriptors, there is no fixed reference value. Instead, we compare the distance of our molecule to the overall chemical space by examining the distances among the molecules in the training set.

We could write code to compute this distance and loop through the training set as we did for the Tanimoto similarities. However, in this case we can take advantage of a scikit-learn function that calculates pairwise distances directly for an entire dataset:

https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.euclidean_distances.html

First, we calculate the Euclidean distance of `X_train` with itself.

This function provides all pairwise distance values as a list of lists, since it allows calculation for more than one prediction at a time. As with the Tanimoto similarity, we do not need to consider all the values — only one is necessary to justify whether our molecule lies within the applicability domain. In this case, we are interested in the maximum distance among the molecules in the training set.


**Hint:** To simplify the calculation of the maximum value in the matrix (a list of lists), you can flatten it into a single list using the `flatten()` method




In [ ]:
#@title Calculate the maximum pairwise distance in the training dataset
from sklearn.metrics.pairwise import euclidean_distances

tr_distances = euclidean_distances(X_train_scaled,X_train_scaled)


max_ed= max(tr_distances.flatten())

print('The maximum value of the Eucledian distance in the training set is ', max_ed)


The maximum value of the Eucledian distance in the training set is  9.000899468120126


The next step is to calculate the maximum Euclidean distance between our target molecule (caffeine) and the training dataset using the same function. If this distance is smaller than the maximum distance within the training set, our molecule is considered to be within the applicability domain.

In your code, include not only the calculation of the maximum Euclidean distance but also a variable named `AD_euclidean` that takes the value `YES` or `NO`, depending on the result.

In [ ]:
#@title Calculate the maximum distance between the target molecule and the training dataset
from sklearn.metrics.pairwise import euclidean_distances

#Distances are calculated in the same way, but by combining the target and training dataframes
caf_distances = euclidean_distances(X_caf_scaled, X_train_scaled)

max_ed_caf= caf_distances.max()

if max_ed_caf <= max_ed:
    AD_eucledian = 'YES' #Assign the text labels IN or OUT accordingly
else:
    AD_eucledian = 'NO' #Assign the text labels IN or OUT accordingly

print('The maximum value of the Euclidean distance between caffeine and the training set is ', max_ed_caf)
print('Is caffeine within the applicability domain of your model?',AD_eucledian)

The maximum value of the Euclidean distance between caffeine and the training set is  8.927112568156375
Is caffeine within the applicability domain of your model? YES


## Section 6: Exercise with multiple molecules

Predict the toxicity of the following molecules using the Earthworm acute toxicity model, and indicate whether each molecule lies within the applicability domain according to the two methods described above.
 * Name: Phenol
 * CAS: 50-78-2
 * SMILES: COCO
 * Name: Nicotine
 * Name: Streptokinase CAS: 9002-01-1



 ### Step 1: Generate a dataframe with a single column named `'SMILES'` and populate it with your target SMILES strings.

In [ ]:
import pandas as pd

target_smiles = [
    "c1ccccc1O", # Phenol
    "CN1CCC(C[N+]1C)C1=CN=CC=C1" # Nicotine
]

df_target = pd.DataFrame({'SMILES': target_smiles})
display(df_target)

,SMILES
0,c1ccccc1O
1,CN1CCC(C[N+]1C)C1=CN=CC=C1


### Step 2: Check the model
* Regenerate the model.

* Recover the `X_train` dataframe containing the non-scaled descriptors from the training set.

* Retrieve the list of selected descriptors as `sel_descriptors`.

This step is not strictly necessary if you still have these variables from the previous sections, but it is good practice to repeat it to ensure that you are working with the correct model.


In [ ]:
#@title Regenerate the model
#Write your code here

from sklearn.linear_model import LogisticRegression

estimator = LogisticRegression(random_state=42, max_iter=1000)
metrics = check_classification_model(estimator,
                           train_file_path,
                           test_file_path,
                           'Final')

train_database = pd.read_csv(train_file_path, sep=';')  #Non-scaled training set

# Assuming sel_descriptors should be the same as train_descs from previous steps
sel_descriptors = train_descs

# Reduce the train_database to include only the selected descriptors
train_database_reduced = train_database[sel_descriptors]
# For consistency with previous steps, database_reduced was created using train_database_noscaled.
# I'll create a new variable for the reduced training database based on the current train_database.

# The descriptors_df variable was used previously to denote scaled descriptors, but it's not present
# as an explicit variable in the kernel state for this part. The `imputer_scaler` function already handles
# scaling from the original database_reduced. So, we'll ensure `database_reduced` is correctly passed
# to the imputer_scaler in the subsequent step.


        TRAIN				TEST
		Neg.	Pos.			Neg.	Pos.
	Negative	34	9		Negative	16	11
	Positive	9	34		Positive	4	7
Accuracy:	 0.79 				 0.61
Sensitivity:	 0.79 				 0.64
Specificity:	 0.79 				 0.59
Precision:	 0.79 				 0.39
Cross validation test accuracy: 0.709 +/- 0.065


### Step 3: Calculate and process the selected descriptors for all target molecules.

Check and adapt the code from Section 2. In this case, the SMILES should be read from df_target, and the selected descriptors should come from the cell above. You may divide the code into several cells if you prefer.

In [ ]:
#@title Calculate the required descriptors
#Write your code here

#Import the required modules and functions (pandas, Mordred, and RDKit)
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors
from mordred import Calculator, descriptors
import numpy as np

# Create a list of mol objects from df_target['SMILES']
mol_obj_list_target = [Chem.MolFromSmiles(s) for s in df_target['SMILES']]

#Calculate the RDKit descriptors
rdkit_desc_data_target = []
for m in mol_obj_list_target:
    if m is not None:
        rdkit_desc_data_target.append(Descriptors.CalcMolDescriptors(m))
    else:
        rdkit_desc_data_target.append({}) # Handle cases where mol conversion fails
df_rdkit_target = pd.DataFrame(rdkit_desc_data_target)

#Concatenate with df_target for now, similar to how df_caf_desc was created
df_target_with_rdkit = pd.concat([df_target, df_rdkit_target], axis=1)

#Calculate the Mordred descriptors
calc = Calculator(descriptors, ignore_3D=True)
mordred_df_target = calc.pandas(mol_obj_list_target)

#Combine the two sets of descriptors
df_target_desc = pd.concat([df_target_with_rdkit, mordred_df_target], axis=1)

#Print the shape of the dataframe to ensure that all descriptors are included
print("\nShape of the final df:",df_target_desc.shape)

# Select only the columns corresponding to the selected descriptors (sel_descriptors)
# Initialize X_target with NaN values for all selected descriptors, then populate
X_target = pd.DataFrame(columns=sel_descriptors, index=range(len(df_target)))
for col in sel_descriptors:
    if col in df_target_desc.columns:
        X_target[col] = df_target_desc[col]
    # If a descriptor from sel_descriptors is not found in df_target_desc, it remains NaN
    # This will then be handled by the KNNImputer

#Run the imputation_prediction function
X_target_scaled = imputer_scaler(X_target, train_database_reduced)


#Keep the print statements to check your work
print("\nX_target (unscaled, with NaNs if any):\n", X_target)
print("\nX_target_scaled:\n", X_target_scaled)

100%|██████████| 2/2 [00:00<00:00, 17.85it/s]


Shape of the final df: (2, 1824)

X_target (unscaled, with NaNs if any):
    BCUT2D_MRHI PEOE_VSA4_rdk PEOE_VSA9_rdk SMR_VSA2_rdk SlogP_VSA8_rdk
0     5.179740           NaN           NaN          NaN            NaN
1     5.154463           NaN           NaN          NaN            NaN

X_target_scaled:
    BCUT2D_MRHI  PEOE_VSA4_rdk  PEOE_VSA9_rdk  SMR_VSA2_rdk  SlogP_VSA8_rdk
0     5.179740      -0.561668        0.45418     -0.306009       -0.457609
1     5.154463      -0.561668        0.45418     -0.306009       -0.457609


### Step 4: Perform the prediction and save the results as a list.

Remember that the predict method reads an entire table, so you can make predictions for all molecules at once.

In [ ]:
#@title Prediction
#Write your code here
Y_target_pred = estimator.predict(X_target_scaled)

# Keep this to view the results
print("Predictions for target molecules:", Y_target_pred)

Predictions for target molecules: [1 1]


### Step 5: Calculate the applicability domain for each molecule using both methods.

 * Tanimoto-Jaccard
 * Euclidean distance

Apply both methods separately, and for each one, obtain a list containing `YES` or `NO` values indicating whether each molecule is within the applicability domain (AD).

In [ ]:
#@title Use the Tanimoto-Jacard method
threshold = 0.573

#Import the necessary modules, if needed
from rdkit import Chem
from rdkit.Chem import rdMolDescriptors
from rdkit import DataStructs

#Get the list of SMILES from the datasets
smi_train = train_database['SMILES']
smi_target = df_target['SMILES']


AD_tanimoto=[]
for smi in smi_target:
    #Create a loop to iterate over each target molecule
    mol = Chem.MolFromSmiles(smi)
    target_fingerprint = rdMolDescriptors.GetMACCSKeysFingerprint(mol)

    max_sim = 0.0
    for i in smi_train:
        #Obtain the Mol object and fingerprint for each molecule in the training dataset
        train_mol = Chem.MolFromSmiles(i)
        if train_mol is not None:
            train_fingerprint = rdMolDescriptors.GetMACCSKeysFingerprint(train_mol)

            #Calculate the similarity score using the FingerprintSimilarity function
            sim_score = DataStructs.FingerprintSimilarity(target_fingerprint, train_fingerprint)
            #Update the variable if the new value is greater than the current maximum
            if sim_score > max_sim:
                max_sim = sim_score

    #At the end, obtain the maximum similarity score
    print(f'Maximum similarity index for {smi}: {round(max_sim,3)}')
    #Compare the result with the threshold and append 'YES' or 'NO' to the list accordingly
    if max_sim >= threshold:
        AD_tanimoto.append('YES')
    else:
        AD_tanimoto.append('NO')

#Keep this to display the list when the code is executed
AD_tanimoto

Maximum similarity index for c1ccccc1O: 0.667
Maximum similarity index for CN1CCC(C[N+]1C)C1=CN=CC=C1: 0.5


['YES', 'NO']

In [ ]:
#@title Use the Euclidean distance method
from sklearn.metrics.pairwise import euclidean_distances

#First, calculate the maximum internal distance within the training dataset to use as a threshold
# max_ed is already calculated in a previous cell and available in the kernel state.
print('The maximum value of the Euclidean distance in the training set is ', max_ed)

AD_eucledian=[]

# Calculate distances between target molecules and training set
target_distances = euclidean_distances(X_target_scaled, X_train_scaled)

# Iterate through each target molecule's distances to the training set
for i, distances_for_target_mol in enumerate(target_distances):
    # Find the maximum distance for the current target molecule
    max_ed_targ = distances_for_target_mol.max()

    print(f'The maximum value of the Euclidean distance between target molecule {i} and the training set is {max_ed_targ}')

    # Compare the result with max_ed and append 'YES' or 'NO' to the list accordingly
    if max_ed_targ <= max_ed:
        AD_eucledian.append('YES')
    else:
        AD_eucledian.append('NO')

#Keep this to check the results
AD_eucledian

The maximum value of the Euclidean distance in the training set is  9.000899468120126
The maximum value of the Euclidean distance between target molecule 0 and the training set is 8.659989972449369
The maximum value of the Euclidean distance between target molecule 1 and the training set is 8.6473701688594


['YES', 'YES']

###**Q3.** Present your results in a dataframe with the following columns: `"SMILES"`, `"Prediction"`, `"Tanimoto"`, and `"Euclidean Distance"`.

Visualise the table using the code below, and then copy it into the platform.


In [ ]:
#@title Save the final table
#Write your code

final_dataframe = pd.DataFrame(df_target['SMILES'],columns=['SMILES'])
final_dataframe['Prediction'] = Y_target_pred
final_dataframe['Tanimoto'] = AD_tanimoto
final_dataframe['Euclidean Distance'] = AD_eucledian

#Optionally, save it to a file for later review

final_dataframe

,SMILES,Prediction,Tanimoto,Euclidean Distance
0,c1ccccc1O,1,YES,YES
1,CN1CCC(C[N+]1C)C1=CN=CC=C1,1,NO,YES
